# Glove vs Fasttext vs Word2Vec

In [ ]:
import pandas as pd
import gensim.downloader as api

from pathlib import Path
from collections import Counter
from nltk.tokenize import TreebankWordTokenizer
from src.utils import load_datasets, PretrainedEmbeddings

In [26]:
train_df, val_df, test_df = load_datasets()

In [28]:
def download_pretrained(option):
    """
    You can use this to download GloVe (or similar) model using wget.
    It's fine to download it through other means, but don't unzip the file (the rest of the notebook can
    handle the file in its zipped format).

    This function returns the path which you can use for the loader class that follows.

    The function will skip the download if it detects the file in disk. If the file in disk is corrupted, you will have to delete the file first.
    """
    if option == "glove_50":
        destination = "glove.6B.50d.txt"
        dims = 50
        skip_first = False

    elif option == "fasttext":
        destination = "wiki-news-300d-1M.vec.zip"
        dims = 300
        skip_first = True

    else:
        raise ValueError(f"Unknown option: {option}")


    if not Path(destination).is_file():

        if option == "glove_50":
            !wget -O glove.6B.50d.txt https://huggingface.co/kcz358/glove/resolve/main/glove.6B/glove.6B.50d.txt

        elif option == "fasttext":
            !wget https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip

        if not Path(destination).is_file():
            raise FileNotFoundError(f"Download failed: {destination}")

    return destination, dims, skip_first

**Embedding model proposals**

In [ ]:
# Compare the vocab size of the models

models = {}

for option in ["glove_50", "fasttext", "word2vec"]:

    if option != "word2vec":
        path, dims, skip_first = download_pretrained(option)

        models[option] = PretrainedEmbeddings(
            path, dims, skip_first_line=skip_first
        )

    else:
        models[option] = api.load("word2vec-google-news-300")

    if option == "word2vec":
        dims = models[option].vector_size
        vocab_size = len(models[option].key_to_index)
    else:
        dims = models[option].num_dims
        vocab_size = len(models[option].vocab)

    print(option, "dimensions:", dims, "vocabulary size:", vocab_size)

glove_50 dimensions: 50 vocabulary size: 400002
fasttext dimensions: 300 vocabulary size: 830463
word2vec dimensions: 300 vocabulary size: 3000000


Word Tokenization for the dataset

In [ ]:
tokenizer = TreebankWordTokenizer()

def tokenize(text):
    tokens = tokenizer.tokenize(text)

    return [
        token for token in tokens
        if any(char.isalnum() for char in token) # removes punctuation (because of word2vec mainly)
    ]

# Example tokenization
for sentence in train_df["sentence1"].head(5):
    print("Original:", sentence)
    print("Tokens:", tokenize(sentence), "\n")

Original: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Tokens: ['In', 'Paris', 'in', 'October', '1560', 'he', 'secretly', 'met', 'the', 'English', 'ambassador', 'Nicolas', 'Throckmorton', 'asking', 'him', 'for', 'a', 'passport', 'to', 'return', 'to', 'England', 'through', 'Scotland'] 

Original: The NBA season of 1975 -- 76 was the 30th season of the National Basketball Association .
Tokens: ['The', 'NBA', 'season', 'of', '1975', '76', 'was', 'the', '30th', 'season', 'of', 'the', 'National', 'Basketball', 'Association'] 

Original: There are also specific discussions , public profile debates and project discussions .
Tokens: ['There', 'are', 'also', 'specific', 'discussions', 'public', 'profile', 'debates', 'and', 'project', 'discussions'] 

Original: When comparable rates of flow can be maintained , the results are high .
Tokens: ['When', 'comparable', 'rates', 'of', 'flow

In [68]:
sentences = pd.concat([
    train_df["sentence1"],
    train_df["sentence2"]
], ignore_index=True)

tokens = [
    token
    for sentence in sentences
    for token in tokenize(sentence)
]

print("Total tokens:", len(tokens))
print("Unique tokens:", len(set(tokens)))

Total tokens: 1824862
Unique tokens: 35294


**Compare vocabulary coverage for the embedding models**

In [ ]:
def vocabulary_coverage(model, tokens):
    unique_tokens = set(tokens)

    covered_total_tokens = sum(token in model for token in tokens)
    covered_unique_tokens = sum(token in model for token in unique_tokens)

    total_token_coverage = covered_total_tokens / len(tokens)
    unique_token_coverage = covered_unique_tokens / len(unique_tokens)

    oov_words = [
        word for word in unique_tokens
        if word not in model
    ]

    return {
        "total_token_coverage": total_token_coverage,
        "unique_token_coverage": unique_token_coverage,
        "oov_words": oov_words
    }

In [ ]:
results = {}

for name, model in models.items():
    results[name] = vocabulary_coverage(model, tokens)

coverage_df = pd.DataFrame({
    name: {
        "Total token coverage": result["total_token_coverage"] * 100,
        "Unique token coverage": result["unique_token_coverage"] * 100
    }
    for name, result in results.items()
}).T

display(coverage_df)

,Token coverage,Type coverage
glove_50,97.913760,87.312291
fasttext,98.602305,91.134470
word2vec,83.471024,81.962940


**Missing token examples**

In [71]:
token_counts = Counter(tokens)

for name, result in results.items():
    print(f"{name}: Most frequent missing tokens")

    missing = sorted(
        result["oov_words"],
        key=lambda word: token_counts[word],
        reverse=True
    )

    for word in missing[:8]:
        print(word, token_counts[word])
    print()

glove_50: Most frequent missing tokens
Berasia 136
Leudesius 48
Garré 48
Shantih 48
Aganbegyan 48
Eoacmaea 48
Eletrônica 48
Urechioiu 48

fasttext: Most frequent missing tokens
Lottiidae 80
Leudesius 48
Aganbegyan 48
Eoacmaea 48
UPFA-SLFP 48
Eletrônica 48
Urechioiu 48
Eoacmaeidae 48

word2vec: Most frequent missing tokens
and 65840
of 60405
a 35172
to 27754
's 5338
2011 1201
2015 1082
2010 954

